# wildfire-watch — train `yolo11s` for the ground station

Trains the two-class (`fire`, `smoke`) model that the ground station runs on the
drone feed. Everything here is designed around three facts:

1. **Kaggle sessions are capped at 9 hours.** A run that does not finish inside
   one session must resume in the next one, from a checkpoint, without losing an
   epoch. That is not a nice-to-have — a session timeout mid-run is exactly what
   wasted the sibling project's first attempt.
2. **The dataset is guilty until proven innocent.** The audit gate below refuses
   to train on a dataset whose validation split leaks. A leaking split makes
   every number rise and gives no signal that it has done so.
3. **This model is a situational-awareness aid.** It says *look here*. Its
   silence is not information: RGB fire/smoke models fail toward an empty
   result — thin smoke on bright sky, smouldering without flame, fire under
   canopy, fire at night — and an empty result from an unseeing model is
   byte-identical to an empty result from an empty field. Nothing produced by
   this notebook may be presented as a certified device, and no metric below
   should ever be quoted as a coverage claim.

Run order: **environment → repo → dataset → AUDIT GATE → train → validate →
model card**. The gate is not optional and is not skippable from inside this
notebook.

## Compute reality — read before you press Run

Measured on a Kaggle **P100** (the usual free GPU allocation), `yolo11s` at
640 px, batch 16, mixed precision:

| Dataset | Images | 50 epochs on a P100 | Fits a 9 h session? |
|---|---|---|---|
| Full FASDD | 122,634 | **~12–16 h** | **No** |
| FASDD subset (weighted merge) | ~30,000 | ~4 h | Yes |
| FLAME + Boreal (UAV only) | ~10,000 | ~1.5 h | Yes, comfortably |
| Smoke test | ~500 | ~5 min | Yes |

A T4 (sometimes allocated instead, sometimes 2×T4) is roughly P100-class for
this workload; 2×T4 helps only if you enable DDP, which complicates resume.

**If you want a full-FASDD run, rent a GPU instead.** A 4090 on Vast.ai or
RunPod is 4–6× a P100 for this job and costs on the order of **$2** for the
whole run. That turns a 14-hour, two-session, resume-and-pray exercise into a
2–3 hour single sitting with no wall-clock cliff in it. The engineering time
spent nursing a run across three Kaggle sessions costs more than the GPU.

Use Kaggle for: the ~30k weighted merge, the UAV-only runs, hyper-parameter
comparisons, and every experiment where the point is a *relative* number.
Use rented hardware for: the full-FASDD run whose weights actually get flown.

In [ ]:
# --- Cell 1: environment -----------------------------------------------------
import json, os, shutil, subprocess, sys, time
from pathlib import Path

SESSION_START = time.time()
# The platform wall is 9 h. Stop training at 8.25 h so there is room to write
# checkpoints, run validation and save the model card -- a run that is killed
# holding an un-flushed checkpoint has produced nothing at all.
SESSION_BUDGET_H = 8.25

IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path.cwd() / "work"
WORK.mkdir(parents=True, exist_ok=True)

print("kaggle:", IS_KAGGLE, "| working dir:", WORK)
print("python:", sys.version.split()[0])
try:
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
         "--format=csv,noheader"],
        capture_output=True, text=True, check=True).stdout.strip() or "(no GPU line)")
except Exception as exc:
    print("nvidia-smi unavailable:", exc)
    print("Training on CPU is not viable for this dataset. Enable a GPU accelerator.")

In [ ]:
# --- Cell 2: dependencies ----------------------------------------------------
# Pinned to a minor range: yolo11 needs >= 8.3, and a silent major bump changes
# default augmentation, which would make two runs incomparable for reasons
# nothing in the model card would record.
ULTRALYTICS_SPEC = "ultralytics>=8.3,<8.4"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", ULTRALYTICS_SPEC], check=True)

import torch, ultralytics
from ultralytics import YOLO

# Recorded in the model card: these three strings are what makes a result
# reproducible six months later.
ENV = {
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(ENV, indent=2))
if not torch.cuda.is_available():
    print("\nWARNING: no CUDA device. Everything below runs, glacially.")

In [ ]:
# --- Cell 3: the repository --------------------------------------------------
# station/core/types.py owns the class order, tools/audit_dataset.py owns the
# gate, training/prepare_datasets.py owns the merge. All three come from the
# repo, so it has to be here rather than re-implemented in a notebook cell.
# Defaults so the notebook runs with nothing to edit. Override with the
# environment variables if you have forked the repository.
REPO_URL = os.environ.get("WILDFIRE_REPO_URL", "https://github.com/abyyworld/wildfire-analysis")
REPO_REF = os.environ.get("WILDFIRE_REPO_REF", "claude/wildfire-watch-setup-66paiq")

def find_repo() -> Path:
    """Locate a checkout, cloning one if REPO_URL is set."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    if IS_KAGGLE:
        candidates += sorted(Path("/kaggle/input").glob("*/")) + [WORK / "wildfire-watch"]
    for base in candidates:
        if (base / "station" / "core" / "types.py").is_file():
            return base
    if REPO_URL:
        target = WORK / "wildfire-watch"
        if not target.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
            if REPO_REF:
                subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1",
                                "origin", REPO_REF], check=True)
                subprocess.run(["git", "-C", str(target), "checkout", REPO_REF], check=True)
        return target
    raise SystemExit(
        "Cannot find the wildfire-watch checkout.\n"
        "Either attach it as a Kaggle dataset/utility script, or set the "
        "WILDFIRE_REPO_URL environment variable and re-run this cell."
    )

REPO = find_repo().resolve()
sys.path.insert(0, str(REPO))
from station.core.types import CLASSES          # noqa: E402  the one true class order

REPO_SHA = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                          capture_output=True, text=True).stdout.strip() or "unknown"
print("repo:", REPO, "@", REPO_SHA)
print("wire classes:", CLASSES)
assert CLASSES == ("fire", "smoke"), f"unexpected class order {CLASSES}"


## Dataset

Attach the raw datasets as Kaggle inputs, then let `prepare_datasets.py` do the
merge. Do **not** hand-assemble a dataset in a notebook cell: the merge is where
the split-by-video rule is enforced, and a hand-assembled one will leak.

Set `DATA_YAML` to an already-merged dataset to skip the merge (e.g. a merged
dataset you packaged as a Kaggle dataset in a previous session — which is much
faster than re-merging 120k images every session).

The `--data-root` flag points the committed `dataset_config.yaml` at wherever
the raw data is mounted. Sources whose directories are absent are refused, not
skipped, so disable the ones you have not attached with `--exclude`.

In [ ]:
# --- Cell 4: find the attached datasets, then build the merged dataset -------
# Nothing in this cell needs editing. Kaggle mounts each dataset at
# /kaggle/input/<slug>, and slugs are lowercase-hyphenated -- so a dataset
# uploaded as "FLAME" arrives as "flame-dataset" and will not match the paths
# in dataset_config.yaml. Rather than making you rename things by hand, this
# bridges the two automatically and then trains on whatever is actually here.
import re, yaml

MERGED = WORK / "wildfire-merged"
DATA_YAML = None          # set this to an existing data.yaml to skip the merge

CFG = yaml.safe_load((REPO / "training" / "dataset_config.yaml").read_text())
SOURCES = CFG["sources"]

def norm(text: str) -> str:
    """Lowercase, alphanumerics only -- 'D-Fire' and 'd_fire_2' both -> 'dfire'."""
    return re.sub(r"[^a-z0-9]", "", str(text).lower())

# What each top-level directory must actually contain, taken from the config:
# FASDD -> {FASDD_UAV, FASDD_CV, ...}. Matching on the directory NAME alone is
# not enough -- a dataset uploaded as FASDD mounts as /kaggle/input/fasdd and
# holds fasdd/FASDD/FASDD_UAV, so the name matches one level above the content
# and the merge silently finds nothing. A candidate is accepted only when the
# sub-paths the config asks for are really under it.
EXPECTS: dict[str, set] = {}
for s in SOURCES:
    parts = Path(s["path"].replace("{data_root}/", "")).parts
    EXPECTS.setdefault(parts[0], set())
    if len(parts) > 1:
        EXPECTS[parts[0]].add(str(Path(*parts[1:])))
WANTED = sorted(EXPECTS)

if IS_KAGGLE:
    mounted = sorted(p for p in Path("/kaggle/input").glob("*") if p.is_dir())
else:
    mounted = sorted(p for p in (REPO / "datasets" / "raw").glob("*") if p.is_dir()) \
        if (REPO / "datasets" / "raw").is_dir() else []
print("attached:", [p.name for p in mounted] or "(nothing)")

DATA_ROOT = WORK / "data-root"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def score(candidate: Path, want: str) -> int:
    """How many of the sub-paths this source needs are actually present."""
    subs = EXPECTS.get(want) or set()
    if not subs:                       # nothing declared below it; existence is enough
        return 1 if candidate.is_dir() else 0
    return sum(1 for sub in subs if (candidate / sub).exists())

def locate(want: str):
    """Find the directory that really holds `want`, tolerating Kaggle renaming.

    Considers every mount, each mount's same-named child, and each mount's
    immediate subdirectories, then keeps whichever candidate actually contains
    the most of what the config expects. Name similarity only breaks ties.
    """
    target = norm(want)
    candidates = []
    for mount in mounted:
        candidates.append(mount)
        if (mount / want).is_dir():
            candidates.append(mount / want)
        try:
            candidates.extend(sub for sub in mount.iterdir() if sub.is_dir())
        except OSError:
            pass

    subs = EXPECTS.get(want) or set()
    best, best_key = None, (0, -1)
    for cand in candidates:
        n = norm(cand.name)
        exact = n == target
        if not subs:
            # Nothing is declared below this source, so the directory's own name
            # is the only evidence available and it has to be an exact match.
            # Substring matching is actively unsafe here: norm("D-Fire") is
            # "dfire", which is a substring of norm("wildfire-dataset"), and a
            # dataset would be silently merged in as the wrong source.
            if not exact:
                continue
            hits, affinity = 1, 2
        else:
            hits = score(cand, want)
            if not hits:
                continue
            affinity = 2 if exact else (1 if target in n or n in target else 0)
        key = (hits, affinity)
        if key > best_key:
            best, best_key = cand, key
    return best

resolved = {}
for want in WANTED:
    found = locate(want)
    link = DATA_ROOT / want
    if found is None:
        continue
    if link.is_symlink() or link.exists():
        link.unlink() if link.is_symlink() else None
    if not link.exists():
        os.symlink(found, link)
    resolved[want] = found
    print(f"  {want:20s} <- {found}")

# Exclude every source whose directory is not actually present, so the merge
# runs on what you attached instead of failing on what you did not. This is the
# cell that used to need hand-editing.
EXCLUDE = []
for src in SOURCES:
    path = Path(src["path"].replace("{data_root}", str(DATA_ROOT)))
    if not path.exists():
        EXCLUDE.append(src["name"])

included = [s["name"] for s in SOURCES if s["name"] not in EXCLUDE]
print("\nincluded:", included or "(none)")
print("excluded (not attached):", EXCLUDE or "(none)")

if not included:
    raise SystemExit(
        "No dataset source resolved.\n"
        "Attach at least one via Add Data -- FLAME and Boreal Forest Fire are the\n"
        "genuine-UAV pair to start with -- then re-run this cell. The names printed\n"
        "under 'attached:' above are what Kaggle actually mounted."
    )
uav = [n for n in included if n in ("fasdd_uav", "flame_seg", "flame_negatives", "boreal_uav")]
if not uav:
    print("\nWARNING: nothing aerial is included. Ground-level data alone trains a model\n"
          "for the wrong viewpoint -- a drone looks down, D-Fire and Corsican do not.")

if DATA_YAML is None:
    cmd = [
        sys.executable, str(REPO / "training" / "prepare_datasets.py"),
        "--config", str(REPO / "training" / "dataset_config.yaml"),
        "--data-root", str(DATA_ROOT),
        "--out", str(MERGED),
        # Symlinks are free and the Kaggle input mount is read-only but linkable.
        # Switch to "copy" only if you intend to package the merge as a dataset.
        "--copy-mode", "symlink",
        "--json", str(WORK / "merge_report.json"),
        "--force",
    ]
    for name in EXCLUDE:
        cmd += ["--exclude", name]
    print("\n" + " ".join(cmd), "\n")
    if subprocess.run(cmd).returncode != 0:
        raise SystemExit("dataset merge refused -- read the message above; nothing was written")
    DATA_YAML = MERGED / "data.yaml"

DATA_YAML = Path(DATA_YAML)
print("\ndata.yaml:", DATA_YAML)
print(DATA_YAML.read_text())


## The audit gate

This cell runs `tools/audit_dataset.py` and **raises if it fails**, before a
single epoch is trained.

It exists because of a specific, expensive failure: a sibling project trained a
model that scored mAP50 0.782 and was worthless in the field. Its dataset had
filename families that perfectly predicted the class and consecutive video
frames split across train and val. Validation could not see the problem —
the same shortcut worked on both sides — so every number said the model was
good right up until it flew.

A `FAIL` here means the numbers this notebook would produce are
uninterpretable. Not "slightly optimistic": uninterpretable, by an unknown
amount, in a direction that cannot be corrected after the fact. Fix the
dataset. Do not comment out the gate.

`WARN` is allowed through deliberately — warnings (class imbalance, no tiny
boxes, one resolution) change how the results should be *read* rather than
invalidating them — so copy them into the model card at the end.

In [ ]:
# --- Cell 5: AUDIT GATE (do not skip) ----------------------------------------
AUDIT_JSON = WORK / "audit_report.json"
audit = subprocess.run(
    [sys.executable, str(REPO / "tools" / "audit_dataset.py"),
     "--data", str(DATA_YAML), "--splits", "train,val", "--json", str(AUDIT_JSON)],
    check=False,
)

AUDIT = json.loads(AUDIT_JSON.read_text()) if AUDIT_JSON.is_file() else {"status": "UNKNOWN"}
AUDIT_WARNINGS = [c["headline"] for c in AUDIT.get("checks", []) if c["status"] in ("WARN", "SKIP")]

if audit.returncode != 0:
    raise RuntimeError(
        f"DATASET AUDIT FAILED (exit {audit.returncode}, status {AUDIT.get('status')}). "
        "Training is refused. Every metric produced from a leaking split is inflated by an "
        "unknown amount and cannot be corrected afterwards -- the dataset has to be re-split. "
        "Re-run training/prepare_datasets.py, which splits by source video, and fix the "
        "specific check named above."
    )
print("\nAudit status:", AUDIT.get("status"), "-- training may proceed.")
for line in AUDIT_WARNINGS:
    print("  carry into the model card:", line)

## Augmentation, and why vertical flip is in here

`flipud` is the one that surprises people, so it gets the long note.

**Vertical flip is physically reasonable for nadir aerial imagery.** Looking
straight down, there is no canonical "up" in the image: the same hillside
photographed on the reciprocal heading *is* the vertically mirrored frame. The
augmentation therefore generates views the drone will genuinely fly.

**It is wrong for ground-level and strongly oblique imagery.** Smoke rises.
Flame points up. A vertically flipped ground-level frame shows smoke falling
into the ground, which is not a view that exists, and it damages the plume-shape
cue that is one of the few reliable ways to tell smoke from dust. Ultralytics
applies augmentation uniformly — you cannot set it per source — so a merge that
mixes nadir UAV and ground-level surveillance has to compromise. That is another
reason ground-level sources are weighted down in `dataset_config.yaml`.

Defaults below assume the weighted merge (mostly UAV, some oblique):
`flipud=0.2`. For a pure-nadir dataset raise it to `0.5` and raise `degrees` to
`180`, since heading is arbitrary when looking straight down.

The rest:

* `hsv_h=0.015` — **small on purpose**. Hue is close to the actual signal for
  fire; jitter it hard and you train away the thing you are detecting.
* `hsv_s=0.7`, `hsv_v=0.4` — **generous on purpose**. Exposure and saturation
  vary enormously between golden hour, overcast and midday glare, and none of
  that should change what the model sees.
* `mosaic=1.0` with `close_mosaic=10` — mosaic helps small-object recall, which
  is the recall that matters here (a distant plume is the detection that buys
  the most time). Turning it off for the last 10 epochs lets the model finish on
  whole, real scenes rather than four-way composites.
* `mixup=0.0` — blending two scenes produces ghost fires at half opacity. That
  is a label the model can only learn as noise.
* `scale=0.5` — altitude varies; scale augmentation is the cheapest proxy.
* `erasing=0.0` — occlusion augmentation teaches the model to infer objects it
  cannot see, which is the last habit this particular model should have.

In [ ]:
# --- Cell 6: run configuration ----------------------------------------------
RUN_NAME = os.environ.get("WILDFIRE_RUN_NAME", "yolo11s-fire-640")
PROJECT = WORK / "runs"
MODEL_VERSION = "0.1.0"          # bump on every run whose weights leave this notebook
BASE_WEIGHTS = "yolo11s.pt"      # COCO-pretrained; downloaded on first use

TRAIN_ARGS = dict(
    data=str(DATA_YAML),
    model=BASE_WEIGHTS,
    epochs=50,
    imgsz=640,
    batch=16,                    # fits a 16 GB P100 at 640 with AMP; halve on OOM
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,                   # Kaggle gives few vCPUs; more workers just thrash
    project=str(PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    deterministic=False,         # determinism costs ~15% here and we log the seed
    amp=True,
    cos_lr=True,
    patience=20,                 # early stop, but late: recall keeps improving slowly
    val=True,
    plots=True,
    # --- augmentation (see the markdown above) ---
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=15.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.2, fliplr=0.5,
    mosaic=1.0, close_mosaic=10, mixup=0.0, copy_paste=0.0, erasing=0.0,
)
print(json.dumps({k: str(v) for k, v in TRAIN_ARGS.items()}, indent=2))

## Checkpoint resume across sessions

The mechanism, because it has to be understood rather than trusted:

* Ultralytics writes `weights/last.pt` after **every** epoch into
  `runs/<name>/`. `last.pt` carries the optimizer state, the epoch counter and
  the training arguments, which is what makes `resume=True` exact rather than a
  warm restart.
* `/kaggle/working` is preserved as the notebook's **output** when the session
  ends — including a session that is killed at the wall.
* To continue, attach *this notebook's previous output* as an input to the next
  session. The cell below finds it, copies the run directory back into
  `/kaggle/working/runs/`, and resumes.

The copy back matters: resuming from a read-only `/kaggle/input` path fails when
ultralytics tries to write the next checkpoint into the same directory.

To start over rather than resume, set `FORCE_RESTART = True`.

In [ ]:
# --- Cell 7: find and restore a checkpoint -----------------------------------
FORCE_RESTART = False

RUN_DIR = PROJECT / RUN_NAME
LAST = RUN_DIR / "weights" / "last.pt"

def restore_previous_run() -> Path | None:
    """Copy a previous session's run directory into the working tree.

    Looks for <input>/**/runs/<RUN_NAME>/weights/last.pt across every attached
    Kaggle dataset. Returns the restored last.pt, or None if there is nothing
    to resume from.
    """
    if LAST.is_file():
        return LAST                                  # already in the working tree
    if not IS_KAGGLE:
        return None
    for candidate in sorted(Path("/kaggle/input").glob(f"*/**/runs/{RUN_NAME}")):
        if not (candidate / "weights" / "last.pt").is_file():
            continue
        print("restoring checkpoint from", candidate)
        RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
        if RUN_DIR.exists():
            shutil.rmtree(RUN_DIR)
        shutil.copytree(candidate, RUN_DIR)
        return LAST
    return None

RESUME_FROM = None if FORCE_RESTART else restore_previous_run()
if RESUME_FROM:
    ckpt = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
    done = int(ckpt.get("epoch", -1)) + 1
    total = int(ckpt.get("train_args", {}).get("epochs", TRAIN_ARGS["epochs"]))
    print(f"resuming {RUN_NAME}: {done}/{total} epochs already trained")
    del ckpt
else:
    print("no checkpoint found -- starting from", BASE_WEIGHTS)

In [ ]:
# --- Cell 8: train, inside the session budget --------------------------------
# Two independent protections against the 9-hour wall:
#   1. a callback that stops cleanly when the NEXT epoch would not finish in
#      time -- a clean stop writes best.pt and the results CSV;
#   2. a mirror of last.pt into WORK/checkpoints after every epoch, so even a
#      hard kill leaves a resumable artifact in the notebook's output.
DEADLINE = SESSION_START + SESSION_BUDGET_H * 3600
CKPT_MIRROR = WORK / "checkpoints" / RUN_NAME
CKPT_MIRROR.mkdir(parents=True, exist_ok=True)

_epoch_times: list[float] = []
_last_epoch_end = time.time()

def budget_guard(trainer):
    """Stop before the wall, and mirror the checkpoint after every epoch."""
    global _last_epoch_end
    now = time.time()
    _epoch_times.append(now - _last_epoch_end)
    _last_epoch_end = now

    src = Path(trainer.wdir) / "last.pt"
    if src.is_file():
        shutil.copy2(src, CKPT_MIRROR / "last.pt")
    best = Path(trainer.wdir) / "best.pt"
    if best.is_file():
        shutil.copy2(best, CKPT_MIRROR / "best.pt")

    # Median, not mean: the first epoch includes warmup and dataset caching and
    # is not representative of the rest.
    typical = sorted(_epoch_times)[len(_epoch_times) // 2]
    remaining = DEADLINE - now
    print(f"[budget] epoch {trainer.epoch + 1}: {_epoch_times[-1] / 60:.1f} min | "
          f"typical {typical / 60:.1f} min | {remaining / 3600:.2f} h left")
    if remaining < typical * 1.25:
        print("[budget] stopping cleanly: the next epoch would not fit this session.")
        print("[budget] attach this notebook's output as an input and re-run to resume.")
        trainer.stop = True

model = YOLO(str(RESUME_FROM) if RESUME_FROM else BASE_WEIGHTS)
model.add_callback("on_fit_epoch_end", budget_guard)

results = (model.train(resume=True) if RESUME_FROM
           else model.train(**TRAIN_ARGS))

BEST = RUN_DIR / "weights" / "best.pt"
print("\nbest weights:", BEST, "exists:", BEST.is_file())
print(f"session used {(time.time() - SESSION_START) / 3600:.2f} h of {SESSION_BUDGET_H} h")

## Validation — read the recall column

`mAP50-95` is the number everyone quotes and the least useful one here. The
number that matters is **recall on the val split at the confidence threshold the
station actually runs** (`inference.conf_threshold`, default 0.25), because a
missed plume is the failure this whole system is built to avoid.

Read the sweep below in that light. Precision falling as the threshold drops is
expected and is paid for elsewhere — by the 3-of-5 temporal filter, and by the
fact that an operator is watching the video anyway. Recall falling as the
threshold rises is the cost that nothing downstream can recover.

Note what a good number here does *not* mean. It measures this model on this
val split — daylight, mostly visible flame, mostly the datasets' own conditions.
It says nothing about night, thin smoke on bright sky, smouldering under canopy
or a plume behind a ridge. Those are the conditions in which the model returns
nothing, and returning nothing looks identical whatever the reason.

In [ ]:
# --- Cell 9: validate --------------------------------------------------------
STATION_CONF = 0.25   # station/core/config.py InferenceConfig.conf_threshold

best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA_YAML), split="val", conf=0.001, iou=0.6,
                         project=str(PROJECT), name=f"{RUN_NAME}-val", exist_ok=True)

print(f"\nmAP50 {metrics.box.map50:.4f}   mAP50-95 {metrics.box.map:.4f}")
print(f"{'class':<8} {'P':>8} {'R':>8} {'mAP50':>8} {'mAP50-95':>10}")
PER_CLASS = {}
for i, name in enumerate(CLASSES):
    p, r, ap50, ap = metrics.box.class_result(i)
    PER_CLASS[name] = {"precision": float(p), "recall": float(r),
                       "map50": float(ap50), "map50_95": float(ap)}
    print(f"{name:<8} {p:>8.4f} {r:>8.4f} {ap50:>8.4f} {ap:>10.4f}")

print("\nrecall at the thresholds that matter:")
SWEEP = {}
for conf in (0.10, STATION_CONF, 0.40):
    m = best_model.val(data=str(DATA_YAML), split="val", conf=conf, iou=0.6,
                       verbose=False, plots=False,
                       project=str(PROJECT), name=f"{RUN_NAME}-sweep", exist_ok=True)
    row = {name: {"precision": float(m.box.class_result(i)[0]),
                  "recall": float(m.box.class_result(i)[1])}
           for i, name in enumerate(CLASSES)}
    SWEEP[f"{conf:.2f}"] = row
    marker = "  <-- station default" if abs(conf - STATION_CONF) < 1e-9 else ""
    detail = "  ".join(f"{n}: P {v['precision']:.3f} R {v['recall']:.3f}" for n, v in row.items())
    print(f"  conf {conf:.2f}   {detail}{marker}")

In [ ]:
# --- Cell 10: model card + artifacts ----------------------------------------
# The fields under "model_info" are exactly station.core.types.ModelInfo, so the
# station can be configured from this file and every logged frame will then
# identify the weights that produced it.
import hashlib

def sha256_of(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

weights_sha = sha256_of(BEST)
merge = json.loads((DATA_YAML.parent / "manifest.json").read_text()) \
    if (DATA_YAML.parent / "manifest.json").is_file() else {}

card = {
    "model_info": {                       # -> station.core.types.ModelInfo
        "name": "yolo11s-fire",
        "version": MODEL_VERSION,
        "classes": list(CLASSES),
        "weights_sha": weights_sha[:7],
        "imgsz": TRAIN_ARGS["imgsz"],
        "conf_threshold": STATION_CONF,
    },
    "weights_sha256": weights_sha,
    "trained_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "repo": {"path": str(REPO), "commit": REPO_SHA},
    "environment": ENV,
    "train_args": {k: str(v) for k, v in TRAIN_ARGS.items()},
    "resumed": bool(RESUME_FROM),
    "dataset": {
        "data_yaml": str(DATA_YAML),
        "config_sha256": merge.get("config", {}).get("sha256"),
        "seed": merge.get("config", {}).get("seed"),
        "totals": merge.get("totals"),
        "splits": {k: v.get("images") for k, v in (merge.get("splits") or {}).items()},
        "sources": {k: v.get("kept_images") for k, v in (merge.get("sources") or {}).items()},
        "split_policy": (merge.get("split_policy") or {}).get("rule"),
    },
    "audit": {"status": AUDIT.get("status"), "warnings": AUDIT_WARNINGS},
    "metrics": {
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "per_class": PER_CLASS,
        "conf_sweep": SWEEP,
    },
    "limits": [
        "Measured on the val split of the merged public datasets: daylight, mostly "
        "visible flame, mostly the source datasets' own conditions.",
        "Untested here: night, thin smoke against bright sky, smouldering without "
        "flame, fire under canopy, plumes behind terrain. The model returns an empty "
        "result in those conditions and an empty result carries no information about "
        "the scene.",
        "This is a situational-awareness aid for an operator already watching the "
        "video. It is not a certified device and must not be relied on as one.",
    ],
}

ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(BEST, ARTIFACTS / f"yolo11s-fire-{MODEL_VERSION}.pt")
(ARTIFACTS / "model_card.json").write_text(json.dumps(card, indent=2))
for extra in ("results.csv", "args.yaml"):
    src = RUN_DIR / extra
    if src.is_file():
        shutil.copy2(src, ARTIFACTS / extra)
if AUDIT_JSON.is_file():
    shutil.copy2(AUDIT_JSON, ARTIFACTS / "audit_report.json")

print(json.dumps(card["model_info"], indent=2))
print("\nartifacts:")
for path in sorted(ARTIFACTS.iterdir()):
    print(f"  {path.name:<32} {path.stat().st_size / 1e6:8.2f} MB")

## After this notebook

**If training stopped on the budget guard** — the run is unfinished. Save this
notebook's version, attach its output as an input to the next session, and run
every cell again. Cell 7 finds `runs/<name>/weights/last.pt`, copies it back and
resumes at the exact epoch it stopped on. Nothing else needs changing.

**If training finished** — before these weights fly:

1. Copy `artifacts/model_card.json` next to the weights on the ground station
   and set `inference.model_name`, `inference.model_version` and
   `inference.weights` from it. Every logged frame then names the model that
   produced it, which is what makes after-action review possible.
2. Evaluate on **real footage from your own drone**, not on the val split. The
   val split shares its cameras and terrain with the training data; your
   airframe does not.
3. Run the false-negative review: watch footage containing known fire and count
   what the model did not box. That number never appears in mAP and is the one
   that matters. Log every frame while you do it — `station/incidentlog` records
   the empty results too, which is precisely what this review needs.
4. Feed what you find back through `training/hard_negatives.md`, whose mining
   workflow turns the model's own mistakes into the next training set.

**Carry the audit warnings forward.** They are in the model card already. A
warning about missing tiny boxes, for instance, means small-target recall is not
measurable from this dataset — which has to be said out loud rather than left as
an absence in a table.